## **Installing the requirements**

In [2]:
%pip install -Uq tiktoken matplotlib kaggle wandb==0.22.0 tpu-info huggingface_hub


[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
!apt-get update && apt-get install git-lfs

Hit:1 http://deb.debian.org/debian bookworm InRelease
Hit:2 http://deb.debian.org/debian bookworm-updates InRelease
Hit:3 http://deb.debian.org/debian-security bookworm-security InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.3.0-1+deb12u1).
0 upgraded, 0 newly installed, 0 to remove and 108 not upgraded.


In [ ]:
# Remove this Token Dude!, it is dangerous, keep it in env
TOKEN = "hf_token"

In [7]:
import jax
jax.devices()

E0000 00:00:1766297108.730731    2445 common_lib.cc:612] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:230


[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0),
 TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0),
 TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0),
 TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0),
 TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0),
 TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0),
 TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0),
 TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]

In [8]:
GLOBAL_SEED = 42
DATA_SEED = GLOBAL_SEED + 1
MODEL_INIT_SEED = GLOBAL_SEED + 2
TRAIN_RNG_SEED = GLOBAL_SEED + 3
VAL_RNG_BASE_SEED = GLOBAL_SEED + 4
GEN_SEED = GLOBAL_SEED + 5

## **Build the JAX MoE Model**

- **Architecture:** Decoder-only MoE Transformer with token+pos embeddings, causal multi-head self-attention, RMSNorm residuals, and per-layer MoE FFNs.  
- **Topology:** 12 transformer blocks (default), D_model=768, num_heads=16, FFN_dim=2048, max_seq_len=512.  
- **Router:** dense gate → noisy top-k selection (train) → renormalized gates; emits an aux load-balancing loss to prevent expert collapse.  
- **Experts:** `ExpertMLP` stores parameters as `[E, in, out]`; each device slices its assigned expert via `lax.axis_index` and runs a 2-layer SiLU FFN with dropout.  
- **Dispatch (all_to_all):** flatten tokens → sort by expert → build fixed-capacity buckets → `lax.all_to_all` exchange → local expert apply → scatter-add back to original positions.  
- **Attention:** q/k/v projections → scaled dot-product with causal mask → softmax → concat heads → output projection + dropout.  
- **Outputs & diagnostics:** returns `(logits, router_loss_total, dropped_fraction)` — monitor `dropped_fraction` for capacity overflow and `router_loss_total` for balance.  
- **Parallelism:** designed for `pmap`/device mesh with `axis_name="device"`; align `NUM_EXPERTS` and sharding with available devices for best efficiency.  
- **Defaults & notes:** VOCAB=50257, DROPOUT=0.0, NUM_EXPERTS=8, TOP_K=2, AUX_LOSS_WEIGHT=0.01; tune `capacity_factor`, `jitter_noise`, and provide RNGs for `"jitter"` and `"dropout"`.


In [9]:
import math
from dataclasses import dataclass
from typing import Any, Tuple

from sympy.strategies.tools import top_down

import jax
import jax.numpy as jnp
from jax import lax
from jax import random
import optax
import flax
from flax import linen as nn
from flax.training.train_state import TrainState
import flax.nnx as nnx
import tiktoken

# Model Hyperparameters (Constants)

VOCAB_SIZE: int = 50257 # GPT-2 vocab size
D_MODEL: int = 768

NUM_LAYERS: int = 12 
NUM_HEADS: int = 16 # Divisible by the number of devices
MAX_SEQ_LEN: int = 512 # 512 tokens per sequence, actual gpt uses 1024, we use 512 for memory reasons
GLOBAL_BATCH = 128
STEPS_PER_EPOCH = 1000
DROPOUT: float = 0.0

# MoE settings
NUM_EXPERTS: int = 8      # Number of experts
TOP_K: int = 2 # Number of experts to use
FFN_DIM: int = 2048

NDEV: int = NUM_EXPERTS
AUX_LOSS_WEIGHT = 0.01 # Router alpha for loss scaling

def assert_divisible(a: int, b: int, msg: str = ""):
    assert a % b == 0, msg or f"{a} must be divisible by {b}"


def scaled_dot_product_attention(q: jnp.ndarray, 
                                k: jnp.ndarray, 
                                v: jnp.ndarray, 
                                mask: jnp.ndarray | None) -> jnp.ndarray:
    # q,k,v: [B_local, T, H_local, Hd]
    scale = 1.0 / math.sqrt(q.shape[-1])
    att = jnp.einsum("bthd,bshd->bhts", q, k) * scale  # [B_local, H_local, T, T]
    if mask is not None:
        # mask: [1, 1, T, T] broadcastable
        att = jnp.where(mask, att, jnp.full_like(att, -1e30))
    att = nn.softmax(att, axis=-1)
    out = jnp.einsum("bhts,bshd->bthd", att, v)  # [B_local, T, H_local, Hd]
    return out


class MultiHeadAttention(nn.Module):
    d_model: int = D_MODEL
    num_heads: int = NUM_HEADS
    dropout: float = DROPOUT
    axis_name: str = "device"

    @nn.compact
    def __call__(self, x: jnp.ndarray, *, train: bool, attn_mask: jnp.ndarray | None) -> jnp.ndarray:
        # x: [B_local, T, D]
        # Use static device count for shape computations to avoid tracers
        assert_divisible(self.d_model, self.num_heads, "d_model must be divisible by num_heads")

        head_dim = self.d_model // self.num_heads

        q = nn.Dense(self.d_model, use_bias=False, name="q_proj")(x)
        k = nn.Dense(self.d_model, use_bias=False, name="k_proj")(x)
        v = nn.Dense(self.d_model, use_bias=False, name="v_proj")(x)

        # Reshape to head dimension as well
        q = q.reshape(x.shape[0], x.shape[1], self.num_heads, head_dim)
        k = k.reshape(x.shape[0], x.shape[1], self.num_heads, head_dim)
        v = v.reshape(x.shape[0], x.shape[1], self.num_heads, head_dim)

        out = scaled_dot_product_attention(q, k, v, attn_mask) # [B_local, T, local_heads_per_device, head_dim]

        # Combining the heads
        out = out.reshape(x.shape[0], x.shape[1], self.d_model) # [B_local, T, D]

        # Output projection
        y = nn.Dense(self.d_model, use_bias = False, name="out_proj")(out)
        y = nn.Dropout(self.dropout)(y, deterministic=not train)

        return y


class Router(nn.Module):
    num_experts: int
    k: int
    jitter_noise: float = 0.01  # Crucial for exploration
    dtype: any = jnp.float32
    axis_name: str = "device"
    aux_loss_weight: float = AUX_LOSS_WEIGHT

    @nn.compact
    def __call__(
        self,
        x: jnp.ndarray,
        *,
        
        train: bool = False,
    ) -> Tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
        
        # 1. Project input to expert logits
        # We use a lower initialization scale to prevent starting with a biased router
        logits = nn.Dense(
            self.num_experts,
            use_bias=False,
            name="gate",
            kernel_init=nn.initializers.normal(stddev=0.01), # Reduced from 0.02
            dtype=self.dtype
        )(x) # [B, T, E]
        # 3. Calculate Softmax (Probabilities)
        # We use the noisy logits for selection, but clean logits for backprop/loss sometimes.
        # ST-MoE recommends simple softmax on noisy logits.
        probs = nn.softmax(logits, axis=-1) # [B, T, E]

        if train:
            noise = jax.random.normal(self.make_rng("jitter"), logits.shape, dtype=logits.dtype)
            logits_for_routing = logits + noise * self.jitter_noise
        else:
            logits_for_routing = logits

        # Select top-k from noisy logits
        _, topk_idx = lax.top_k(logits_for_routing, self.k)

        # Selecting clean probs for weight computation
        topk_vals = jnp.take_along_axis(probs, topk_idx, axis=-1)

        # 5. Renormalize the Gates
        denominator = jnp.sum(topk_vals, axis=-1, keepdims=True) + 1e-6
        gates = topk_vals / denominator # [B, T, k]
        
        # A. Load Balancing Loss (Switch Transformer style)
        # importance: The "soft" routing weight (what the router *wanted* to do)
        importance = jnp.mean(probs, axis=(0, 1)) 
        
        # load: The "hard" routing decisions (what the router *actually* did)
        # We create a one-hot of the selected indices
        one_hot_selection = jax.nn.one_hot(topk_idx, self.num_experts, dtype=self.dtype) # [B, T, K, E]
        selection_count = jnp.sum(one_hot_selection, axis=-2) # Sum over K -> [B, T, E]
        load = jnp.mean(selection_count, axis=(0, 1))
        
        # Standard Aux loss: discourages expert collapse, compute similarity between importance and load
        aux_loss = self.num_experts * jnp.dot(importance, load) / self.k
        total_router_loss = self.aux_loss_weight * aux_loss
        return topk_idx, gates, total_router_loss
        

# Expert MLP (one per device) and MoE dispatch via all_to_all

class ExpertMLP(nn.Module):
    d_model: int = D_MODEL
    hidden_dim: int = FFN_DIM
    dropout: float = DROPOUT
    num_experts: int = NUM_EXPERTS
    axis_name: str = "device"

    @nn.compact
    def __call__(self, x: jnp.ndarray, *, train: bool) -> jnp.ndarray:
        # x: [N, D] tokens destined for this specific device/expert
        
        # 1. Identify which Expert ID this device is responsible for
        # This allows us to maintain a single "Bank" of parameters for checkpointing,
        # but only load/use the specific slice relevant to this device.
        expert_id = lax.axis_index(self.axis_name) % self.num_experts

        # 2. Define Initializers (Standard JAX/Flax init)
        # kernel_init: Glorot/Xavier Uniform is standard for Transformers
        kernel_init = nn.initializers.glorot_uniform()
        bias_init = nn.initializers.zeros_init()

        # 3. Create Parameter Bank [Num_Experts, In, Out]
        # JAX initializers automatically handle the 3D shape correctly (treating [E] as batch)
        
        # Layer 1: d_model -> hidden_dim
        w1_all = self.param(
            "w1", 
            kernel_init, 
            (
                self.num_experts, 
                self.d_model, 
                self.hidden_dim
            )
        )

        b1_all = self.param(
            "b1", 
            bias_init, 
            (
                self.num_experts, 
                self.hidden_dim
            )
            )
        
        # Layer 2: hidden_dim -> d_model
        w2_all = self.param(
            "w2", 
            kernel_init, 
            (
                self.num_experts, self.hidden_dim, self.d_model
            )
        )

        b2_all = self.param(
            "b2", 
            bias_init, 
            (
                self.num_experts, 
                self.d_model
            )
        )

        # 4. Slice parameters for this specific device
        # This is a cheap view operation, not a copy
        w1 = w1_all[expert_id] 
        b1 = b1_all[expert_id]
        w2 = w2_all[expert_id]
        b2 = b2_all[expert_id]

        # 5. Feed Forward Computation
        # w1 projection
        h = jnp.matmul(x, w1) + b1
        
        # Activation
        h = nn.silu(h) 
        
        # Dropout 1
        h = nn.Dropout(self.dropout)(h, deterministic=not train)
        
        # w2 projection
        y = jnp.matmul(h, w2) + b2
        
        # Dropout 2
        y = nn.Dropout(self.dropout)(y, deterministic=not train)
        
        return y


def expert_dispatch_a2a(
    x: jnp.ndarray,
    topk_idx: jnp.ndarray,
    gates: jnp.ndarray, # For computing G(x) * E(x)
    expert_apply_fn,
    axis_name: str = "device",
    train: bool = True,
) -> Tuple[jnp.ndarray, jnp.ndarray]:

    B_local, T, D = x.shape
    ndev = NDEV  # static number of devices/experts
    k = int(topk_idx.shape[-1])

    # Flatten token assignments
    N = B_local * T 
    x_flat = x.reshape(N, D) # eg: [[254, 256, 276, 5, 0, 623], [......], [......], ...] -> [254, 256, 276, 5, 0, 623, ...]
    expert_ids = topk_idx.reshape(N * k)  # [N*k] -> eg: [0, 2, 1, 1, 4, 5, 4, 8, 1, 0...] Which expert to process the token
    gate_vals = gates.reshape(N * k)      # [N*k] -> Simply the gate values corresponding to expert_ids

    # Build source indices for reconstruction
    tok_ids = jnp.arange(N)
    tok_ids = jnp.repeat(tok_ids, k)      # [N*k]
    # Recover (b_idx, t_idx)
    b_idx = tok_ids // T
    t_idx = tok_ids % T

    # Sort by destination expert to build per-destination buckets
    sort_idx = jnp.argsort(expert_ids) # sort the indices by expert id
    expert_ids_sorted = expert_ids[sort_idx] # [N*k] -> eg: [0, 0, 1, 1, 1, 2, 4, 4, 5, 8, ...]
    gate_vals_sorted = gate_vals[sort_idx]

    x_sorted = x_flat[jnp.repeat(jnp.arange(N), k)][sort_idx]
    b_sorted = b_idx[sort_idx] # Sort the batch indices 
    t_sorted = t_idx[sort_idx] # Sort the token indices

    # Determine ranges per expert id in the sorted list
    all_devices = jnp.arange(NDEV) # [0, 1, 2, ...]

    # Determine the position indices of where each expert id starts and ends in the sorted list
    starts = jnp.searchsorted(expert_ids_sorted, all_devices, side="left") # eg: [0, 2, 5, 7, 9, ...] -> for example, 1 starts at index 2 
    ends = jnp.searchsorted(expert_ids_sorted, all_devices, side="right") # eg: [2, 5, 7, 9, ...] -> and ends at index 5
    sizes = ends - starts  # eg: [2, 3, 2, 1, ...] Gets the total size of each bucket

    # Capacity per (sender->dest) bucket: average assignments per expert with factor
    # For each expert, we will determine how many tokens at max it can process.
    # In some cases, the router will route more tokens to an expert than the capacity.
    # In that case, we will drop the tokens that overflow the capacity.
    # Eg: N = 128, k = 2, ndev = 8 -> capacity = 128 * 2 / 8 = 32 tokens max per expert
    # Compute capacity as a Python int to keep jnp.arange argument concrete

    capacity_factor = 1.25
    capacity = max(1, int(math.ceil((N * k) / ndev * capacity_factor)))


    def make_bucket(start, size):
        idx = jnp.arange(capacity)
        src_indices = start + idx
        mask = idx < size
        src_indices = jnp.where(mask, src_indices, 0)
        x_vals = x_sorted[src_indices]
        g_vals = gate_vals_sorted[src_indices]
        b_vals = b_sorted[src_indices]
        t_vals = t_sorted[src_indices]
        # Zero-out padded rows
        x_vals = jnp.where(mask[:, None], x_vals, 0.0)
        g_vals = jnp.where(mask, g_vals, 0.0)
        b_vals = jnp.where(mask, b_vals, 0)
        t_vals = jnp.where(mask, t_vals, 0)
        return x_vals, g_vals, b_vals, t_vals, mask

    # buckets for x, gates, batch indices, token indices, and mask
    x_buckets, g_buckets, b_buckets, t_buckets, m_buckets = jax.vmap(make_bucket)(starts, sizes)
    
    # x_buckets = [ndev, capacity, D] Each expert gets a bucket of tokens with max capacity
    # g_buckets = [ndev, capacity] Each expert gets a bucket of gates with max capacity
    # b_buckets = [ndev, capacity] Each expert gets a bucket of batch indices with max capacity
    # t_buckets = [ndev, capacity] Each expert gets a bucket of token indices with max capacity
    # m_buckets = [ndev, capacity] Each expert gets a bucket of mask with max capacity

    # Round 1: send per-destination buckets to their destination expert/device
    def a2a(arr):
        return lax.all_to_all(arr, axis_name=axis_name, split_axis=0, concat_axis=0)

    x_recv = a2a(x_buckets)  # [ndev*capacity, D] on each device, all destined to local expert
    g_recv = a2a(g_buckets)  # [ndev*capacity]
    b_recv = a2a(b_buckets)  # [ndev*capacity]
    t_recv = a2a(t_buckets)  # [ndev*capacity]
    m_recv = a2a(m_buckets)  # [ndev*capacity]

    # Compute local expert output on received tokens
    x_recv = x_recv.reshape(-1, D)
    g_recv = g_recv.reshape(-1)
    b_recv = b_recv.reshape(-1)
    t_recv = t_recv.reshape(-1)
    m_recv = m_recv.reshape(-1)

    y_local = expert_apply_fn(x_recv, train=train)  # [M, D] 
    y_local = y_local * g_recv[:, None] # G(x) * E(x)
    y_local = jnp.where(m_recv[:, None], y_local, 0.0) # Mask out padded rows

    # Reshape to [ndev, capacity, D] per sender (implicit sender order in all_to_all)
    y_per_sender = y_local.reshape(ndev, capacity, D)
    b_per_sender = b_recv.reshape(ndev, capacity)
    t_per_sender = t_recv.reshape(ndev, capacity)
    m_per_sender = m_recv.reshape(ndev, capacity)

    # Round 2: send results back to source devices (by sender index)
    y_ret = a2a(y_per_sender)  # [ndev*capacity, D] now on source devices
    b_ret = a2a(b_per_sender)  # [ndev*capacity]
    t_ret = a2a(t_per_sender)  # [ndev*capacity]
    m_ret = a2a(m_per_sender)  # [ndev*capacity]

    # Scatter-add to output
    y_out = jnp.zeros((B_local, T, D), dtype=x.dtype)
    y_ret = y_ret.reshape(-1, D)
    b_ret = b_ret.reshape(-1)
    t_ret = t_ret.reshape(-1)
    m_ret = m_ret.reshape(-1)
    y_out = y_out.at[b_ret, t_ret, :].add(jnp.where(m_ret[:, None], y_ret, 0.0))

    # Report simple drop fraction (if any bucket overflowed, we clipped by capacity)
    kept_per_expert = jnp.minimum(sizes, capacity)
    kept_assign = jnp.sum(kept_per_expert)
    dropped_fraction = 1.0 - (kept_assign / (N * k))
    return y_out, dropped_fraction


class Block(nn.Module):
    d_model: int = D_MODEL
    num_heads: int = NUM_HEADS
    ffn_dim: int = FFN_DIM
    num_experts: int = NUM_EXPERTS
    top_k: int = TOP_K
    dropout: float = DROPOUT
    axis_name: str = "device"

    @nn.compact
    def __call__(
        self, x: jnp.ndarray, 
        *, 
        train: bool, 
        attn_mask: jnp.ndarray | None) -> Tuple[jnp.ndarray, jnp.ndarray]:
        
        h = nn.RMSNorm()(x)

        h_attn = MultiHeadAttention(
            self.d_model, 
            self.num_heads, 
            self.dropout, 
            axis_name=self.axis_name)(h, train=train, attn_mask=attn_mask)
        x = x + h_attn # Residual connection

        # RMSNorm
        h = nn.RMSNorm()(x)

        # Router
        topk_idx, gates, router_loss = Router(
            num_experts=self.num_experts, 
            k=self.top_k, 
            )(h, train=train)

        # Expert
        expert = ExpertMLP(
            self.d_model, 
            self.ffn_dim, 
            self.dropout, 
            self.num_experts, 
            axis_name=self.axis_name
        )

        expert_apply = lambda inp, train: expert(inp, train=train)

        # Dispatch tokens to experts living in multiple devices
        e_out, dropped_frac = expert_dispatch_a2a(
            h, 
            topk_idx, 
            gates, 
            expert_apply, 
            axis_name=self.axis_name, 
            train=bool(train)
        )

        x = x + e_out # Residual connection
        return x, router_loss, dropped_frac


class MoE(nn.Module):
    vocab_size: int = VOCAB_SIZE
    d_model: int = D_MODEL
    num_heads: int = NUM_HEADS
    num_layers: int = NUM_LAYERS
    max_seq_len: int = MAX_SEQ_LEN
    num_experts: int = NUM_EXPERTS
    top_k: int = TOP_K
    ffn_dim: int = FFN_DIM
    dropout: float = DROPOUT
    axis_name: str = "device"


    @nn.compact
    def __call__(self, 
                input_ids: jnp.ndarray, 
                *, 
                train: bool) -> Tuple[jnp.ndarray, jnp.ndarray]:
        # input_ids: [B_local, T]
        B, T = input_ids.shape
        
        token_emb = nn.Embed(
            self.vocab_size, 
            self.d_model, 
            name="tok_emb", 
        )(input_ids) # Token embeddings

        pos_indices = jnp.arange(T)[None, :]
        pos_emb = nn.Embed(
            self.max_seq_len, 
            self.d_model, 
            name="pos_emb", 
        )(pos_indices) # Position embeddings

        x = token_emb + pos_emb # Add token and position embeddings
        x = nn.Dropout(self.dropout)(x, deterministic=not train)
        # causal mask: [1, 1, T, T]
        mask = jnp.tril(jnp.ones((1, 1, T, T), dtype=bool))
        router_loss_total = jnp.array(0.0, dtype=jnp.float32)
        dropped_frac_sum = jnp.array(0.0, dtype=jnp.float32)
        for _ in range(self.num_layers):
            x, router_loss, dropped_frac = Block(
                self.d_model,
                self.num_heads,
                self.ffn_dim,
                self.num_experts,
                self.top_k,
                self.dropout,
                axis_name=self.axis_name
            )(x, train=train, attn_mask=mask)
            router_loss_total += router_loss
            dropped_frac_sum += dropped_frac
        x = nn.RMSNorm()(x)
        logits = nn.Dense(self.vocab_size, use_bias=False, name="lm_head")(x)
        return logits, router_loss_total, dropped_frac_sum

## **Data Preprocessing & Batch generation**

In [ ]:
import numpy as np
import os
import glob

data_dir = "/kaggle/input/openwebtext-gpt2"

train_tokens = np.memmap(os.path.join(data_dir, "train.bin"), dtype=np.uint16, mode="r")
val_tokens = np.memmap(os.path.join(data_dir, "val.bin"), dtype=np.uint16, mode="r")


# Verify
print(f"Train tokens shape: {train_tokens.shape}") # Should be (Total_Tokens,)
print(f"Val tokens shape: {val_tokens.shape}")

Train tokens shape: (9035582489,)
Val tokens shape: (4434606,)


Creates a global batch of contiguous token sequences for language modeling by randomly sampling start positions, vectorized gathering `[batch, seq_len]` slices, and reshaping them into per-device local batches for `pmap`. Ensures `GLOBAL_BATCH` is evenly divisible across devices (`NDEV`), supports optional seeded RNG for reproducibility, and returns device-sharded token tensors ready for next-token prediction (inputs `[:-1]`, targets `[1:]`).

In [11]:

def get_batch(split: str = "train", seq_len: int = MAX_SEQ_LEN, global_batch: int = GLOBAL_BATCH, rng=None):
    data = train_tokens if split == "train" else val_tokens
    num_devices = jax.local_device_count()
    assert num_devices == NDEV
    assert global_batch % num_devices == 0
    b_local = global_batch // num_devices

    max_start = data.shape[0] - seq_len
    assert max_start > 0, f"Data too short ({data.shape[0]}) for seq_len={seq_len}"

    # Use host RNG if none provided (optionally seedable)
    if rng is None:
        starts = np.random.randint(0, max_start, size=(global_batch,))
    else:
        # if a numpy-like rng (np.random.Generator) is provided:
        starts = rng.integers(0, max_start, size=(global_batch,))

    # Vectorized gather: build indices (GLOBAL_BATCH x seq_len) and index once
    offsets = np.arange(seq_len)[None, :]          # shape [1, seq_len]
    indices = starts[:, None] + offsets            # shape [GLOBAL_BATCH, seq_len]
    batch_np = data[indices]                       # vectorized gather

    tokens = jnp.asarray(batch_np, dtype=jnp.int32)           # [GLOBAL_BATCH, seq_len]
    tokens = tokens.reshape(num_devices, b_local, seq_len)    # [num_devices, b_local, seq_len]

    # Standard LM inputs/targets: input = tokens[..., :-1], label = tokens[..., 1:]

    return tokens

## **Training Pipeline**

In [12]:
import os

# Training Constants
PEAK_LR = 1e-3
WARMUP_STEPS = 2000 # Reduced from 5000 for faster convergence check
TOTAL_STEPS = 90000 # Total number of training steps, GPT-2
VALIDATION_STEPS = 500 # Step at which to validate
CKPT_DIR = "/kaggle/working/checkpoints"
CHECKPOINT_EVERY = 5000 # 6 checkpoints will be saved.
GENERATION_STEPS = 1000 # Generate text in every 1000 steps for validation

print(CKPT_DIR)

os.makedirs(CKPT_DIR, exist_ok=True)

/kaggle/working/checkpoints


### **Save and Restore Checkpoints**

In [13]:
from flax.jax_utils import unreplicate, replicate
from orbax.checkpoint import Checkpointer, CheckpointManager, PyTreeCheckpointHandler
import shutil


def save_checkpoint(state, train_rngs, step, ckpt_dir="/kaggle/working/checkpoints"):
    # Make sure checkpoint directory exists
    os.makedirs(ckpt_dir, exist_ok=True)
 
    host_state = unreplicate(state) # Unreplicate the state to get the host-side template
    host_state = jax.device_get(host_state) # Move the state to the host
    host_rng = jax.device_get(train_rngs[0]) # Move the RNG to the host

    payload = {
        "train_state": host_state,
        "train_rng": np.asarray(host_rng),
    }

    # Create a step-specific checkpoint path
    step_dir = os.path.join(ckpt_dir, f"step_{step}")

    # Orbax checkpointer for PyTrees
    checkpointer = Checkpointer(PyTreeCheckpointHandler())

    # Save checkpoint
    checkpointer.save(step_dir, payload)
    print(f"Checkpoint saved at step {step} -> {step_dir}")
    
    # Clean up old checkpoints to save space
    try:
        for item in os.listdir(ckpt_dir):
            item_path = os.path.join(ckpt_dir, item)
            # Delete if it's a directory, starts with 'step_', and is NOT the current step
            if os.path.isdir(item_path) and item.startswith("step_") and item != f"step_{step}":
                print(f"Deleting old checkpoint: {item}")
                shutil.rmtree(item_path)
    except Exception as e:
        print(f"Warning: Failed to clean up old checkpoints: {e}")



def restore_checkpoint(ckpt_dir, step=None):
    # Find available checkpoints
    if step is None:
        # Find the latest checkpoint
        ckpt_dirs = [d for d in os.listdir(ckpt_dir) if d.startswith("step_")]
        if not ckpt_dirs:
            raise ValueError(f"No checkpoints found in {ckpt_dir}")
        
        # Extract step numbers and find the max
        steps = [int(d.split("_")[1]) for d in ckpt_dirs]
        step = max(steps)
    
    step_dir = os.path.join(ckpt_dir, f"step_{step}")
    
    if not os.path.exists(step_dir):
        raise ValueError(f"Checkpoint directory {step_dir} does not exist")
    
    print(f"Restoring checkpoint from {step_dir}")
    
    # Create Orbax checkpointer
    checkpointer = Checkpointer(PyTreeCheckpointHandler())
    
    # Restore the checkpoint
    payload = checkpointer.restore(step_dir)
    
    state = payload["train_state"]
    train_rng = jax.random.PRNGKey(TRAIN_RNG_SEED)  # Initialize with dummy
    
    # Extract RNG if it exists
    if "train_rng" in payload:
        train_rng_array = payload["train_rng"]
        # Convert back to JAX PRNGKey
        train_rng = jnp.array(train_rng_array, dtype=jnp.uint32)
    
    print(f"Successfully restored checkpoint from step {step}")
    return state, train_rng, step
   

### **JAX State Management**

Defines a linear warmup followed by cosine decay learning-rate schedule using Optax, then builds a `TrainState` with AdamW and global-norm gradient clipping. Model parameters are initialized via `MoE.init` when not provided, using per-device RNGs for parameters, dropout, and router jitter. State creation is parallelized with `pmap` to ensure identical optimizer and parameter initialization across devices. A pmapped inference function applies the model in eval mode to return full-sequence logits, enabling efficient multi-device inference while keeping post-processing on the host.


In [14]:
# Linear Warmup + Cosine Decay schedule
def make_schedule(peak_lr: float, warmup_steps: int, total_steps: int, end_lr_ratio: float = 0.1):
    decay_steps = max(1, total_steps - warmup_steps)
    end_lr = peak_lr * end_lr_ratio
    
    return optax.warmup_cosine_decay_schedule(
        init_value=0.0,           # start at 0.0
        peak_value=peak_lr,       # warm up to this value
        warmup_steps=warmup_steps,
        decay_steps=decay_steps,  # cosine portion length
        end_value=end_lr,         # final LR after cosine decay completes
    )


def create_state_function(
    rngs,
    tokens_local,
    *,
    peak_lr,
    warmup_steps,
    total_steps,
    params=None,
):
    model = MoE()

    lr_schedule = make_schedule(peak_lr, warmup_steps, total_steps)

    tx = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(
            learning_rate=lr_schedule,
            weight_decay=0.01,
        ),
    )

    if params is None:
        variables = model.init(rngs, tokens_local, train=True)
        params = variables["params"]

    return TrainState.create(
        apply_fn=model.apply,
        params=params,
        tx=tx,
    )



def pmap_create_state(per_device_key, tokens_local):
    rngs = {'params': per_device_key, 'dropout': per_device_key, 'jitter': per_device_key}
    return create_state_function(
        rngs, 
        tokens_local,
        params = None,
        peak_lr = PEAK_LR,
        warmup_steps = WARMUP_STEPS,
        total_steps = TOTAL_STEPS,
        
    )

def _infer_logits(state, tokens, rng):
    """
    Returns logits for full sequence under pmap; we will use the last position on host.
    """
    # Capture metrics to avoid code changes; we don't use them here.
    logits, _, _ = state.apply_fn(
        {"params": state.params},
        tokens,
        train=False,
    )
    
    return logits


# Parallelize the state creation and inference functions
p_create_state = jax.pmap(pmap_create_state, axis_name="device")
p_infer_logits = jax.pmap(_infer_logits, axis_name="device")

### **Train Step Function**

Implements a pmapped training step that runs a forward pass with dropout and router jitter RNGs, computes next-token cross-entropy, and adds the MoE router auxiliary loss. Gradients are computed with `value_and_grad`, averaged across devices via `pmean`, and expert-bank gradients are explicitly rescaled to compensate for all-to-all routing. Global metrics (loss, CE, router loss, dropped fraction) are reduced across devices, parameters are updated with the optimizer, and the updated state, RNG, and logging metrics are returned for synchronous multi-device training.


In [ ]:
import jax
import jax.numpy as jnp
from jax import random
import optax


def train_step(state, rng, tokens):
    rng, step_rng = random.split(rng)
    dropout_rng, jitter_rng = random.split(step_rng)

    def loss_fn(params):
        # Forward pass: model returns logits and *average* auxiliary loss across layers
        logits, router_loss, dropped_frac_sum = state.apply_fn(
            {"params": params},
            tokens,
            rngs={"dropout": dropout_rng, "jitter": jitter_rng},
            train=True,
        )

        # Shift for language modeling
        logits_next = logits[:, :-1, :]
        targets = tokens[:, 1:]

        # Compute cross entropy loss
        ce = optax.softmax_cross_entropy_with_integer_labels(logits_next, targets).mean()

        # Combine main loss + auxiliary loss (already averaged across layers)
        loss = ce + router_loss

        aux_info = {
            "ce": ce,
            "router_loss": router_loss,
            "dropped_fraction_sum": dropped_frac_sum,
        }
        return loss, aux_info

    # Compute gradients
    (loss, aux_info), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)

    # Average grads and metrics across devices
    grads = jax.lax.pmean(grads, axis_name="device")

    # Expert parameters already saw tokens from all devices because of routing via all-to-all
    # Pmean will scale down the gradients by the number of devices,
    # This is needed for other parameters of the network like MHA, Embedding, etc, but for Experts
    # that are living in multiple devices and seeing all the tokens
    # we need to detect the expert gradients and scale them up by the number of devices
    
    def rescale_expert_bank(g):
        if hasattr(g, "shape") and g.ndim > 0 and g.shape[0] == NUM_EXPERTS:
            return g * NDEV # scale up the gradients
        return g

    grads = jax.tree_util.tree_map(rescale_expert_bank, grads)

    # Global means
    loss = jax.lax.pmean(loss, axis_name="device")
    ce = jax.lax.pmean(aux_info["ce"], axis_name="device")
    router_loss = jax.lax.pmean(aux_info["router_loss"], axis_name="device")
    dropped_fraction_sum = jax.lax.pmean(aux_info["dropped_fraction_sum"], axis_name="device")

    # Optimizer update
    new_state = state.apply_gradients(grads=grads)

    # Metrics for logging
    metrics = {
        "loss": loss,
        "ce": ce,
        "router_loss": router_loss,
        "dropped_fraction_sum": dropped_fraction_sum,
    }

    return new_state, rng, metrics


# Parallelize across devices
p_train_step = jax.pmap(train_step, axis_name="device", donate_argnums=(0,))

### **Eval Step Function**

In [17]:
def eval_step(state, rng, tokens):
    # Split RNG per device for the (still-required) router noise key
    rng, jitter_rng = random.split(rng)

    # Forward pass in eval mode;
    logits, router_loss, _= state.apply_fn(
        {"params": state.params},
        tokens,
        train=False,
    )

    # Next-token loss
    logits_next = logits[:, :-1, :]
    targets = tokens[:, 1:]
    ce = optax.softmax_cross_entropy_with_integer_labels(logits_next, targets).mean()
    loss = ce + router_loss

    # Average across devices
    loss = jax.lax.pmean(loss, axis_name="device")
    ce = jax.lax.pmean(ce, axis_name="device")
    router_loss = jax.lax.pmean(router_loss, axis_name="device")

    return {"val_loss": loss, "val_ce": ce, "val_router_loss": router_loss}

p_eval_step = jax.pmap(eval_step, axis_name="device")

### **Generate Text**

Implements constrained autoregressive text generation with nucleus (top-p) sampling over a top-k shortlist, temperature scaling, repetition penalty, and no-repeat-ngram blocking. Logits are produced via pmapped inference, sliced at the last position, then modified on the host to enforce decoding constraints. Tokens are sampled stochastically with a seeded PRNG, appended step-by-step while respecting context length limits, and generation stops early if an EOS token is encountered.



In [18]:
def apply_repetition_penalty(logits: jnp.ndarray, token_ids: list[int], penalty: float) -> jnp.ndarray:
    if penalty is None or penalty <= 1.0 or len(token_ids) == 0:
        return logits
    uniq = sorted(set(int(t) for t in token_ids))
    idx = jnp.array(uniq, dtype=jnp.int32)
    vals = logits[idx]
    # HF-style repetition penalty:
    # if logit > 0 -> divide, else multiply
    vals = jnp.where(vals > 0, vals / penalty, vals * penalty)
    logits = logits.at[idx].set(vals)
    return logits


def get_banned_tokens_no_repeat_ngram(token_ids: list[int], n: int) -> list[int]:
    if n is None or n <= 0:
        return []
    if len(token_ids) < n - 1:
        return []
    prefix = tuple(token_ids[-(n - 1):])
    banned = set()
    # find all earlier occurrences of this prefix and ban their following token
    for i in range(len(token_ids) - n + 1):
        if tuple(token_ids[i : i + n - 1]) == prefix:
            banned.add(int(token_ids[i + n - 1]))
    return sorted(banned)


def sample_top_p_from_top_k(
    logits: jnp.ndarray,
    *,
    key: jax.Array,
    temperature: float = 1.0,
    top_p: float = 0.9,
    top_k: int = 200,
) -> jnp.ndarray:
    # Greedy mode
    if temperature is None or temperature <= 0:
        return jnp.argmax(logits).astype(jnp.int32)
    # Safety
    top_p = float(top_p) if top_p is not None else 1.0
    top_p = min(max(top_p, 0.0), 1.0)
    vocab = logits.shape[-1]
    if top_k is None or top_k <= 0 or top_k > vocab:
        top_k = vocab
    # Work only on top-k candidates for speed
    topk_vals, topk_idx = lax.top_k(logits, top_k)
    topk_vals = topk_vals / float(temperature)
    probs = jax.nn.softmax(topk_vals, axis=-1)
    # Sort candidates by prob desc (within top-k)
    order = jnp.argsort(-probs)
    probs_sorted = probs[order]
    idx_sorted = topk_idx[order]
    cdf = jnp.cumsum(probs_sorted, axis=-1)
    # Keep tokens until cumulative prob exceeds top_p; always keep at least 1 token
    keep = cdf <= top_p
    keep = keep.at[0].set(True)
    probs_filt = jnp.where(keep, probs_sorted, 0.0)
    probs_filt = probs_filt / (jnp.sum(probs_filt) + 1e-20)
    choice = random.categorical(key, jnp.log(probs_filt + 1e-20))
    return idx_sorted[choice].astype(jnp.int32)



def generate_text_v2(
    state,
    prompt_tokens_1d: jnp.ndarray,
    *,
    max_new_tokens: int = 64,
    temperature: float = 0.7,
    top_p: float = 0.9,
    top_k: int = 200,
    repetition_penalty: float = 1.10,
    no_repeat_ngram_size: int = 3,
    eos_token_id: int | None = None,
    seed: int = GEN_SEED,
    max_context_len: int = MAX_SEQ_LEN,
):
    num_devices = jax.local_device_count()
    assert num_devices == NDEV, "Device count must match NDEV"
    prompt_tokens_1d = jnp.asarray(prompt_tokens_1d, dtype=jnp.int32)
    all_ids = [int(x) for x in np.array(prompt_tokens_1d)]
    host_key = random.PRNGKey(seed)
    dummy_rngs = random.split(host_key, num_devices)  # _infer_logits ignores rng, but keep signature stable
    for _ in range(max_new_tokens):
        # context window (prevents pos_emb OOB if you ever generate long)
        ctx = all_ids[-max_context_len:]
        ctx_arr = jnp.asarray(ctx, dtype=jnp.int32)
        tokens = jnp.broadcast_to(ctx_arr[None, None, :], (num_devices, 1, ctx_arr.shape[0]))  # [D,1,T]
        logits = p_infer_logits(state, tokens, dummy_rngs)   # [D,1,T,V]
        last_logits = logits[0, 0, -1]                       # [V]
        # Apply constraints on logits (host-driven constraints, applied as indexed updates)
        banned = get_banned_tokens_no_repeat_ngram(all_ids, int(no_repeat_ngram_size) if no_repeat_ngram_size else 0)
        if banned:
            banned_idx = jnp.asarray(banned, dtype=jnp.int32)
            last_logits = last_logits.at[banned_idx].set(-1e10)
        last_logits = apply_repetition_penalty(last_logits, all_ids, float(repetition_penalty))
        host_key, sk = random.split(host_key)
        next_id = int(sample_top_p_from_top_k(
            last_logits,
            key=sk,
            temperature=float(temperature),
            top_p=float(top_p),
            top_k=int(top_k) if top_k is not None else None,
        ))
        all_ids.append(next_id)
        if eos_token_id is not None and next_id == int(eos_token_id):
            break
    return jnp.asarray(all_ids, dtype=jnp.int32)

### **Weights & Biases Config**

In [19]:

import wandb

wandb.login()  # only needed once per machine/session

wandb.init(
    project="moe-transformer",
    name="jax-moe-run",
    config={
        "peak_lr": PEAK_LR,
        "warmup_steps": WARMUP_STEPS,
        "total_steps": TOTAL_STEPS,
        "global_batch": GLOBAL_BATCH,
        "num_experts": NUM_EXPERTS,
        "max_seq_len": MAX_SEQ_LEN,
    },
)

wandb: Currently logged in as: sidharthan261 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


### **Full Training Function**

Implements the full multi-device training loop with optional checkpoint resume, synchronized state initialization, and pmapped updates across experts/devices. Batches are sampled on the host, sharded per device, and trained with `p_train_step`, while metrics are reduced, logged, and timed each step. Periodic validation, autoregressive text generation for qualitative checks, and checkpoint saving are integrated, with careful buffer synchronization and memory cleanup to control peak usage during large MoE training.


In [ ]:
import time
from flax.training import checkpoints, common_utils as flax_common_utils
from flax.jax_utils import replicate, unreplicate
from flax import serialization
import gc


enc = tiktoken.get_encoding("gpt2")


RESUME_FROM_STEP = None
LATEST_CKPT_DIR = "/kaggle/working/"

def train(resume: bool = False, resume_step: int | None = None):
    num_devices = jax.local_device_count()
    assert num_devices == NDEV == NUM_EXPERTS
    assert GLOBAL_BATCH % num_devices == 0

    data_rng = np.random.default_rng(DATA_SEED)
    val_data_rng = np.random.default_rng(VAL_RNG_BASE_SEED)

    if resume:
        host_raw_state, train_rng, start_step = restore_checkpoint(
            CKPT_DIR, step=resume_step
        )

        init_batch = get_batch("train", rng=data_rng)
        init_keys = random.split(random.PRNGKey(MODEL_INIT_SEED), num_devices)

        # Create state and drop init buffers immediately
        state = p_create_state(init_keys, init_batch)
        state = state.replace(params=None, opt_state=None)

        # Keep checkpoint data on host
        ckpt_params = jax.tree_util.tree_map(
            lambda x: np.asarray(x, dtype=np.float32),
            host_raw_state["params"]
        )

        # Inject params + optimizer state together (minimize peak)
        state = state.replace(
            params=replicate(jax.device_put(ckpt_params)),
        )

        state = state.replace(
            step=replicate(jnp.asarray(start_step, dtype=jnp.int32)),
        )

        train_rngs = random.split(train_rng, num_devices)

        # Synchronize on ALL device buffers
        jax.tree_util.tree_map(lambda x: x.block_until_ready(), state.params)

        # Cleanup
        del host_raw_state, ckpt_params

        print(f"Warm-started from params and optimizer state at step {start_step}")


    
    else:
        init_batch = get_batch("train", rng=data_rng)
        init_keys = random.split(random.PRNGKey(MODEL_INIT_SEED), num_devices)
        state = p_create_state(init_keys, init_batch)
        train_rngs = random.split(random.PRNGKey(TRAIN_RNG_SEED), num_devices)
        start_step = 0

    log_every = 100
    for step in range(start_step + 1, TOTAL_STEPS + 1):
        t0 = time.time()
        tokens = get_batch("train", rng=data_rng)
        state, train_rngs, metrics = p_train_step(state, train_rngs, tokens)
        jax.block_until_ready(metrics["loss"])
        dt = time.time() - t0

        loss = float(metrics["loss"][0])
        ce = float(metrics["ce"][0])
        router_loss = float(metrics["router_loss"][0])
        dropped_sum = float(metrics["dropped_fraction_sum"][0])

        if step % log_every == 0 or step == 1:
            wandb.log(
                {
                    "train/loss": loss,
                    "train/ce": ce,
                    "train/router_loss": router_loss,
                    "train/drop_sum": dropped_sum,
                    "train/time_per_step": dt,
                },
                step=step,
            )
            print(
                f"step {step:06d} | time {dt:.3f}s | loss {loss:.4f} | ce {ce:.4f} | "
                f"router_loss {router_loss:.6f} | dropped sum {dropped_sum:.6f}"
            )

        if step % VALIDATION_STEPS == 0 or step == 100:
            val_tokens = get_batch("val", rng = val_data_rng)
            val_rngs = random.split(random.PRNGKey(VAL_RNG_BASE_SEED + step), num_devices)
            val_metrics = p_eval_step(state, val_rngs, val_tokens)
            jax.block_until_ready(val_metrics["val_loss"])
            val_loss = float(val_metrics["val_loss"][0])
            val_ce = float(val_metrics["val_ce"][0])
            val_router_loss = float(val_metrics["val_router_loss"][0])
            wandb.log(
                {
                    "val/loss": val_loss,
                    "val/ce": val_ce,
                    "val/router_loss": val_router_loss,
                },
                step=step,
            )

            print(
                f"[val] step {step:06d} | val_loss {float(val_metrics['val_loss'][0]):.4f} | "
                f"val_ce {float(val_metrics['val_ce'][0]):.4f} | "
                f"val_router_loss {float(val_metrics['val_router_loss'][0]):.6f}"
            )

        if step % GENERATION_STEPS == 0:

            prompt = "Write a short news article about dual nature of matter.\n\n"
            prompt_ids = jnp.array(enc.encode(prompt), dtype=jnp.int32)
            gen_ids = generate_text_v2(
            state,
            prompt_ids,
            max_new_tokens=100,
            temperature=0.9,
            top_p=0.95,
            top_k=50,
            repetition_penalty=1.15,
            no_repeat_ngram_size=4,
            eos_token_id=50256,     # important
            seed=GEN_SEED + 76000,  # vary seed per step so you don't always sample the same “style”
        )

            generated_text = enc.decode(np.array(gen_ids).tolist())
            print("\n\nGenerated text:\n\n", generated_text)


        if step % CHECKPOINT_EVERY == 0 or step == TOTAL_STEPS:
            save_checkpoint(state, train_rngs, step)
            
            # upload_folder(
            #         repo_id=repo_id,
            #         repo_type="model",
            #         folder_path=CKPT_DIR,
            #         path_in_repo=".",
            #         commit_message=f"Sync checkpoint at step {step}",
            #         ignore_patterns=[".git", ".ipynb_checkpoints"]
            #     )
            # print(f"Checkpoint saved at step {step} and pushed to Hugging Face")
            

### **Let's Train**

In [ ]:

# Can also load with optimizer states as well
train(resume=False)